In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "mps"

# 1. 特征提取器
model = torchvision.models.resnet18(weights='IMAGENET1K_V1')
feature_extractor = torch.nn.Sequential(*list(model.children())[:-1])
feature_extractor.to(device).eval()

# 2. 数据预处理
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=64, shuffle=False)

# 3. 提取特征
features, labels = [], []
with torch.no_grad():
    for imgs, lbls in tqdm(loader):
        imgs = imgs.to(device)
        feat = feature_extractor(imgs).view(imgs.size(0), -1)
        features.append(feat.cpu().numpy())
        labels.append(lbls.numpy())
features = np.concatenate(features)
labels = np.concatenate(labels)

# 4. t-SNE 可视化
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embedded = tsne.fit_transform(features[:2000])
plt.scatter(embedded[:, 0], embedded[:, 1], c=labels[:2000], cmap='tab10', s=5)
plt.show()

  0%|          | 328k/170M [00:19<2:44:48, 17.2kB/s] 


KeyboardInterrupt: 